# Automated E-commerce Reporting System Live Demo

This notebook runs the full Pass 1 pipeline end-to-end on a public sample. It downloads its own data, validates it, cleans it, aggregates to monthly revenue, reconciles against the raw source, and writes an Excel workbook.

Disclosure: Simulated portfolio demonstration. Not a real client engagement.

## Run the pipeline

Click the play button or press Shift+Enter. Wait about 60 seconds. The final cell will write `monthly_revenue.xlsx` and `monthly_revenue.csv` into `/content/report_output/`.

In [ ]:
import os
import pandas as pd

BASE = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"
print("Downloading sample data...")
orders   = pd.read_csv(BASE + "olist_orders_dataset.csv",
                       dtype={"order_id": str, "customer_id": str})
payments = pd.read_csv(BASE + "olist_order_payments_dataset.csv",
                       dtype={"order_id": str, "payment_type": str})
print(f"  orders   : {len(orders):,}")
print(f"  payments : {len(payments):,}")

REQUIRED = {
    "orders":   ["order_id", "order_status", "order_purchase_timestamp"],
    "payments": ["order_id", "payment_sequential", "payment_value"],
}
def validate(df, name):
    missing = [c for c in REQUIRED[name] if c not in df.columns]
    if missing:
        raise RuntimeError(f"{name}: missing required column(s): {missing}")
validate(orders, "orders")
validate(payments, "payments")
print("  validation : PASS")

orders = orders.copy()
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], format="%Y-%m-%d %H:%M:%S", errors="raise")
orders["order_purchase_month"] = orders["order_purchase_timestamp"].dt.strftime("%Y-%m")

pay_pos = payments[payments["payment_value"] > 0]
pay_by_order = (pay_pos.groupby("order_id", as_index=False)["payment_value"]
                .sum().rename(columns={"payment_value": "revenue"}))

merged = orders[["order_id", "order_status", "order_purchase_month"]].merge(
    pay_by_order, on="order_id", how="left")
merged["revenue"] = merged["revenue"].fillna(0.0)

EXCLUDE = {"canceled", "unavailable"}
mask = merged["order_status"].isin(EXCLUDE)
merged["revenue"] = merged["revenue"].where(~mask, 0.0)

print(f"  clean rows : {len(merged):,}")

monthly = (merged.groupby("order_purchase_month", as_index=False)
           .agg(order_count         = ("order_id", "size"),
                orders_with_revenue = ("revenue",  lambda s: int((s > 0).sum())),
                revenue             = ("revenue",  "sum")))
monthly["revenue_per_order"] = (monthly["revenue"] / monthly["order_count"]).round(2)
monthly = monthly.sort_values("order_purchase_month").reset_index(drop=True)

print(f"  monthly rows : {len(monthly)}")
print(f"  total revenue: {monthly['revenue'].sum():,.2f} BRL")
print(f"  total orders : {int(monthly['order_count'].sum()):,}")

rebuilt_rev = merged["revenue"].sum()
diff = abs(rebuilt_rev - monthly["revenue"].sum())
print(f"  rebuild diff : {diff:.6f}")
assert diff < 0.01, "Reconciliation failed"

OUT_DIR = "/content/report_output"
os.makedirs(OUT_DIR, exist_ok=True)
xlsx_path = os.path.join(OUT_DIR, "monthly_revenue.xlsx")
csv_path  = os.path.join(OUT_DIR, "monthly_revenue.csv")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    monthly.to_excel(writer, sheet_name="Monthly Revenue", index=False)

monthly.to_csv(csv_path, index=False)

print()
print("=" * 70)
print("Files written")
print("=" * 70)
print(f"  {xlsx_path}  ({os.path.getsize(xlsx_path):,} bytes)")
print(f"  {csv_path}   ({os.path.getsize(csv_path):,} bytes)")
print()
print("Download them from the Files panel on the left,")
print("or run: from google.colab import files; files.download(xlsx_path)")

print()
print("=" * 70)
print("Monthly revenue report")
print("=" * 70)
print(monthly.to_string(index=False))
print()
print("PASS 1 PIPELINE END TO END VERIFIED")


## What just happened

Every stage you watched has a matching source file in `src/`:

- Load: `src/load_data.py`
- Validate: `src/validate_data.py`
- Clean: `src/clean_data.py`
- Aggregate: `src/analyze_data.py`
- Report: `src/generate_report.py`
- Orchestrate: `src/main.py`

The reconciliation step rebuilt the revenue total from the raw payments and orders without touching the aggregated table. It matched to zero drift.

## To run on your own data

1. Clone the repo
2. Drop two CSVs into `data/input/`
3. Run `python src/main.py --input data/input --output output/runs/<timestamp>`

If your CSVs have the wrong shape, the program refuses to run and tells you which column is missing. It will never silently produce a wrong report.